# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

# Inisialisasi path proyek (kompatibel dari root maupun subfolder notebooks/)
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'wikipedia_id.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: wikipedia_id.parquet
Ukuran file: 311.0 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [2]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'docid': String, 'title': String, 'text': String})


jumlah_baris,docid_null,title_null,text_null
u32,u32,u32,u32
1301683,0,0,0


In [3]:
contoh_data = lazy_data.head(10).collect()
contoh_data

docid,title,text
str,str,str
"""1269713#0""","""Toxophora coeruleiventris""","""Toxophora coeruleiventris adal…"
"""1470508#3""","""Ahmad Hanafi Rais""","""Pada periode 2014-2019, Hanafi…"
"""2378820#12""","""Dapson""","""Hepatitis toksik dan cholestat…"
"""40427#25""","""Tentara Pembebasan Nasional Za…","""The EZLN placed since the begi…"
"""130916#6""","""Universitas Bung Karno""","""Memasuki akhir tahun 2012, UBK…"
"""349432#2""","""Janus (satelit)""","""Massa Janus diperkirakan (1.89…"
"""256167#2""","""Kesultanan Bima""","""Sultan Nuruddin Abubakar Ali S…"
"""1243631#1""","""Mesoleptogaster trimaculata""","""Lalat ini mempunyai insting pr…"
"""2721062#56""","""Pelabuhan Antwerpen""","""Cagar alam ini memiliki area p…"


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [4]:
connection = duckdb.connect()
try:
    profil_duckdb = connection.sql(
        'SUMMARIZE SELECT * FROM read_parquet(?)', params=[str(DATA_PATH)]
    ).pl()
except AttributeError:
    profil_duckdb = connection.execute(
        'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
    ).pl()
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[9,2]"
"""docid""","""VARCHAR""","""1#1""","""999993#1""",1294011,null,null,null,null,null,1301683,0.00
"""title""","""VARCHAR""","""!!! (album)""","""ㅑ""",423902,null,null,null,null,null,1301683,0.00
"""text""","""VARCHAR""","""!""","""😍""",1222805,null,null,null,null,null,1301683,0.00


## Catatan Milestone 1

### 1. Informasi dan Karakteristik Dataset
- **Sumber Data:** Hugging Face [`indonesian-nlp/wikipedia-id`](https://huggingface.co/datasets/indonesian-nlp/wikipedia-id)
- **Nama File Mentah:** `wikipedia_id.parquet` (tersimpan di `data/raw/` tanpa modifikasi)
- **Format:** Apache Parquet (Columnar Storage)
- **Ukuran File:** **311.0 MB** (terkompresi Snappy) / uncompressed memory **> 1 GB**
- **Jumlah Baris:** **1.301.683 baris** (memenuhi syarat tugas: > 1.000.000 baris)
- **Jumlah Kolom:** 3 kolom (`docid`, `title`, `text`)
- **Checksum SHA-256:** `b3f827439ba3a31048e5cae4a93292d59489e34a37aeb758219e94ee0d551c7f`

### 2. Tipe Data dan Skema Kolom
- `docid` (`String` / `VARCHAR`): ID unik dokumen artikel Wikipedia.
- `title` (`String` / `VARCHAR`): Judul artikel ensiklopedia Wikipedia Bahasa Indonesia.
- `text` (`String` / `VARCHAR`): Isi teks artikel lengkap yang telah dibersihkan dari markup wikitext.

### 3. Analisis Missing Values (Null)
- Berdasarkan eksekusi `ringkasan` Polars dan `profil_duckdb`:
  - `docid_null`: 0 (0.00%)
  - `title_null`: 0 (0.00%)
  - `text_null`: 0 (0.00%)
- Tidak terdapat nilai `null` eksplisit pada seluruh kolom dalam dataset.

### 4. Anomali & Rencana Penanganan pada Milestone 2 (Data Cleaning)
1. **Deteksi Nilai Kosong Terselubung (Empty / Whitespace Strings):** Meskipun nilai `null` adalah 0, perlu diuji apakah ada artikel dengan teks kosong `""` atau hanya spasi/newline `"   "`.
2. **Artikel Sangat Pendek (Stub / Disambiguasi):** Terdapat artikel yang hanya berisi beberapa patah kata definisi yang perlu diidentifikasi dan ditangani.
3. **Noise Karakter dan Tanda Baca Ekstrem:** Normalisasi karakter non-alfanumerik atau simbol khusus pada judul dan teks.
4. **Pemeriksaan Duplikasi:** Menguji apakah ada artikel duplikat pada judul (`title`) atau teks (`text`).
5. **Feature Engineering Statistik:** Menambahkan fitur turunan seperti `char_length` (panjang karakter) dan `word_count` (jumlah kata) untuk analisis kualitas pada Milestone 2 dan analisis lanjutan pada Milestone 3.